In [0]:
# O QUE: Criar o schema Gold.
# POR QUÊ: Armazenar as tabelas analíticas.

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

print("Schema Gold disponível!")

In [0]:
# O QUE: Relacionar pedidos com seus respectivos itens.
# POR QUÊ: Preparar a tabela fato de vendas.

from pyspark.sql import functions as F

df_pedidos = spark.table("workspace.silver.orders")
df_itens = spark.table("workspace.silver.order_items")

df_vendas = df_itens.join(
    df_pedidos.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    ),
    on="order_id",
    how="left"
)

print("Total de itens:", df_itens.count())
print("Total após relacionamento:", df_vendas.count())

display(df_vendas.limit(10))

In [0]:
# O QUE: Preparar as colunas da fato de vendas.
# POR QUÊ: Criar uma estrutura organizada para análises.

df_fato_vendas = df_vendas.select(
    "order_id",
    "order_item_id",
    "customer_id",
    "product_id",
    "seller_id",
    "order_status",
    F.to_date("order_purchase_timestamp").alias("data_venda"),
    F.col("price").alias("valor_produto"),
    F.col("freight_value").alias("valor_frete")
)

display(df_fato_vendas.limit(10))

In [0]:
# O QUE: Validar e publicar a fato de vendas.
# POR QUÊ: Garantir a integridade dos dados na Gold.

tabela = "workspace.gold.fato_vendas"

total = df_fato_vendas.count()

assert total == 112650, "Quantidade de registros incorreta"

# Verificar pedidos sem correspondência
sem_cliente = df_fato_vendas.filter(
    F.col("customer_id").isNull()
).count()

assert sem_cliente == 0, "Existem itens sem pedido correspondente"

# Criar a tabela somente se ainda não existir
if not spark.catalog.tableExists(tabela):
    df_fato_vendas.write.format("delta").saveAsTable(tabela)
    print("Fato de vendas criada!")
else:
    print("Tabela já existe. Nenhuma alteração realizada.")

print("Registros:", total)

In [0]:
# O QUE: Dimensão de clientes, Preparar e publicar os clientes na Gold.

df_dim_cliente = spark.table("workspace.silver.customers")

df_dim_cliente = df_dim_cliente.select(
    "customer_id",
    "customer_unique_id",
    "customer_city",
    "customer_state",
    "customer_zip_code_prefix"
)

tabela = "workspace.gold.dim_cliente"

if not spark.catalog.tableExists(tabela):
    df_dim_cliente.write.format("delta").saveAsTable(tabela)
    print("Dimensão criada!")
else:
    print("Dimensão já existe. Preservada.")

In [0]:
# O QUE: Dimensão de produtos, Relacionar produtos e categorias e publicar na Gold.

produtos = spark.table("workspace.silver.products")
categorias = spark.table(
    "workspace.silver.product_category_name_translation"
)

df_dim_produto = (
    produtos
    .join(categorias, on="product_category_name", how="left")
    .select(
        "product_id",
        "product_category_name",
        "product_category_name_english",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
        "product_photos_qty"
    )
)

# Validar a quantidade de produtos
total = df_dim_produto.count()
assert total == 32951, "Quantidade de produtos incorreta"

# Publicar sem sobrescrever tabelas existentes
tabela = "workspace.gold.dim_produto"

if not spark.catalog.tableExists(tabela):
    df_dim_produto.write.format("delta").saveAsTable(tabela)
    print("Dimensão de produtos criada!")
else:
    print("Dimensão já existe. Preservada.")

print("Produtos:", total)

In [0]:
# O QUE: Dimensão de vendedores, Preparar e publicar os vendedores.
# POR QUÊ: Permitir análises de vendas por vendedor e região.

df_dim_vendedor = spark.table("workspace.silver.sellers")

df_dim_vendedor = df_dim_vendedor.select(
    "seller_id",
    "seller_city",
    "seller_state",
    "seller_zip_code_prefix"
)

tabela = "workspace.gold.dim_vendedor"

if not spark.catalog.tableExists(tabela):
    df_dim_vendedor.write.format("delta").saveAsTable(tabela)
    print("Dimensão de vendedores criada!")
else:
    print("Dimensão já existe. Preservada.")

print("Vendedores:", df_dim_vendedor.count())

In [0]:
# O QUE: Dimensão de tempo, Criar um calendário a partir das datas dos pedidos.
# POR QUÊ: Permitir análises por ano, mês e trimestre.

from pyspark.sql import functions as F

datas = df_fato_vendas.select("data_venda").where(
    F.col("data_venda").isNotNull()
)

periodo = datas.agg(
    F.min("data_venda").alias("inicio"),
    F.max("data_venda").alias("fim")
).first()

df_dim_tempo = (
    spark.sql(
        f"SELECT explode(sequence(to_date('{periodo['inicio']}'), "
        f"to_date('{periodo['fim']}'), interval 1 day)) AS data"
    )
    .select(
        "data",
        F.year("data").alias("ano"),
        F.month("data").alias("mes"),
        F.quarter("data").alias("trimestre")
    )
)

tabela = "workspace.gold.dim_tempo"

if not spark.catalog.tableExists(tabela):
    df_dim_tempo.write.format("delta").saveAsTable(tabela)
    print("Dimensão de tempo criada!")
else:
    print("Dimensão já existe. Preservada.")

print("Datas:", df_dim_tempo.count())